In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, copy
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from global_land_mask import globe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import norm, spearmanr, linregress
from scipy import ndimage

# ---- Paths ----
TIF   = rf"{DATA_ROOT}/AIS_processing/traffic_density/world_bank/shipdensity_global.tif"
ACC   = rf"{DATA_ROOT}/climate_AIS_data/accident/accident_parsed.csv"
CELLS = rf"{DATA_ROOT}/climate_AIS_data/accident/grid_1deg_cells.csv"
EGRID = CELLS.replace(".csv", "_Egrid.npy")
NGRID = CELLS.replace(".csv", "_Ngrid.npy")
FIGDIR = rf"{DATA_ROOT}/climate_AIS_data/accident/figs"
os.makedirs(FIGDIR, exist_ok=True)

# ---- Constants ----
YEARS = 74 / 12                 # 2015-01~2021-02 cumulative -> annual mean
NLON, NLAT = 360, 180
STRIP = 200
EXT = [-180, 180, -90, 90]
FLOOR = 87600                   # analysis-domain floor (annual AIS message count)
YR_MIN, YR_MAX = 2002, 2021
DPI = 500                       # ★ publication resolution

# ---- Save two versions ----
# v1 annotated: all text (title/axis labels/legend/stats box/colorbar label)
# v2 clean:     no explanatory text, tick numbers only (for scale alignment when adding text in Keynote)
def save_two(draw_fn, name):
    for annot, tag in [(True, "annotated"), (False, "clean")]:
        fig = draw_fn(annot)
        fig.savefig(os.path.join(FIGDIR, f"{name}_{tag}.png"), dpi=DPI, bbox_inches="tight")
        if annot:
            plt.show()
        plt.close(fig)
    print(f"saved {name}_annotated/clean.png (dpi={DPI}) -> {FIGDIR}")

FORCE_REBUILD = False           # True = delete cache and force re-aggregation
if FORCE_REBUILD:
    for f in (EGRID, NGRID, CELLS):
        if os.path.exists(f):
            os.remove(f); print("removed:", f)


In [ ]:
def build_traffic_grid():
    E = np.zeros(NLAT * NLON); N = np.zeros(NLAT * NLON)
    with rasterio.open(TIF) as src:
        nod = src.nodata if src.nodata is not None else 2147483647
        H, W = src.height, src.width
        t = src.transform; xres, x0, yres, y0 = t.a, t.c, t.e, t.f
        cb = np.clip(np.floor(x0 + (np.arange(W) + 0.5) * xres + 180).astype(int), 0, NLON - 1)
        for r0 in range(0, H, STRIP):
            nr = min(STRIP, H - r0)
            a  = src.read(1, window=Window(0, r0, W, nr)).ravel().astype("float64")
            rb = np.clip(np.floor(y0 + (r0 + np.arange(nr) + 0.5) * yres + 90).astype(int), 0, NLAT - 1)
            flat = (rb[:, None] * NLON + cb[None, :]).ravel()
            v = (a != nod) & (a >= 0) & (a < 1e9)          # drop pathological pixels (static emitters/saturated values)
            E += np.bincount(flat[v], weights=a[v], minlength=NLAT * NLON)
            N += np.bincount(flat[v],               minlength=NLAT * NLON)
            print(f"  rows {r0 + nr}/{H}", end="\r")
    print()
    return E.reshape(NLAT, NLON), N.reshape(NLAT, NLON)

if os.path.exists(EGRID):
    E_grid = np.load(EGRID); N_grid = np.load(NGRID); print("loaded grid cache")
else:
    print("aggregating raster (~a few minutes)...")
    E_grid, N_grid = build_traffic_grid()
    np.save(EGRID, E_grid); np.save(NGRID, N_grid); print("saved grid cache")


In [ ]:
# Check: total vs vessel-hour budget (decisive evidence that unit = AIS messages, for SI)
total = E_grid.sum()
budget_vh = 1e5 * 8760 * 74 / 12
print(f"raster total {total:.2e} vs global vessel-hour budget {budget_vh:.2e} -> ratio {total/budget_vh:.0f}x")


In [ ]:
lat_c = np.arange(NLAT) - 90 + 0.5
lon_c = np.arange(NLON) - 180 + 0.5
LonC, LatC = np.meshgrid(lon_c, lat_c)

# Ocean mask: cell is ocean if any sub-point is ocean (keeps coasts/ports, drops inland false positives)
ocean = np.zeros((NLAT, NLON), bool)
for dla in (-0.35, 0.0, 0.35):
    for dlo in (-0.35, 0.0, 0.35):
        ocean |= globe.is_ocean(LatC + dla, LonC + dlo)

E_annual_full = E_grid / YEARS
domain = ocean & (E_annual_full >= FLOOR)               # ★ the single analysis domain
print(f"domain cells (ocean & E>={FLOOR}): {int(domain.sum())}")

acc = pd.read_csv(ACC, low_memory=False)
la = pd.to_numeric(acc["lat_dec"], errors="coerce")
lo = pd.to_numeric(acc["lon_dec"], errors="coerce")
yr = pd.to_numeric(acc["acc_year"], errors="coerce")
ok = la.notna() & lo.notna() & yr.between(YR_MIN, YR_MAX)
lon_b = np.clip(np.floor(lo[ok].values + 180).astype(int), 0, NLON - 1)
lat_b = np.clip(np.floor(la[ok].values + 90).astype(int),  0, NLAT - 1)
A_grid = np.bincount(lat_b * NLON + lon_b, minlength=NLAT * NLON).reshape(NLAT, NLON)

n_in  = int(A_grid[domain].sum())
n_all = int(A_grid[ocean].sum())
print(f"accidents {YR_MIN}-{YR_MAX}: with coords {int(ok.sum())}, "
      f"in-domain {n_in}, excluded {n_all - n_in} (low-traffic/land)")

rr, cc = np.where(domain)
cells = pd.DataFrame({
    "lat_bin": rr, "lon_bin": cc, "lat_c": lat_c[rr], "lon_c": lon_c[cc],
    "E": E_grid[rr, cc] / YEARS, "A": A_grid[rr, cc].astype(int), "npix": N_grid[rr, cc],
})
cells.to_csv(CELLS, index=False, encoding="utf-8-sig")
print("saved analysis table:", CELLS, "| rows:", len(cells))


In [ ]:
# ================= Block 5: main PPML fit + EB shrinkage =================
df = cells.copy()
df["logE"] = np.log(df["E"])
ppml = smf.glm("A ~ logE", data=df, family=sm.families.Poisson()).fit(cov_type="HC1")
beta, a0 = ppml.params["logE"], ppml.params["Intercept"]
se_b     = ppml.bse["logE"]
p_beta   = 2 * (1 - norm.cdf(abs((beta - 1) / se_b)))
df["mu"] = np.asarray(ppml.fittedvalues)

n_zero = int((df["A"] == 0).sum())
print(f"PPML: beta={beta:.3f} (SE {se_b:.3f}), p(beta=1)={p_beta:.2g}")
print(f"sample {len(df)} cells, cells with A=0 {n_zero} ({n_zero/len(df):.0%}) <- must be >0")

A_, mu_ = df["A"].to_numpy(float), df["mu"].to_numpy(float)
inv_alpha = max((((A_ - mu_) ** 2 - mu_).sum()) / (mu_ ** 2).sum(), 1e-6)
alpha = 1.0 / inv_alpha
df["oe_eb"] = (A_ + alpha) / (mu_ + alpha)
print(f"alpha={alpha:.3f} | raw O/E: {(A_/mu_).min():.2f}~{(A_/mu_).max():.0f}"
      f" | after EB: {df.oe_eb.min():.2f}~{df.oe_eb.max():.1f}")


In [ ]:
# ================= Block 5b: goodness of fit + structural variance decomposition (run before panel c) =================
try:
    mcfadden = 1 - ppml.llf / ppml.llnull
except Exception:
    mcfadden = 1 - ppml.deviance / ppml.null_deviance
dev_r2 = 1 - ppml.deviance / ppml.null_deviance
print(f"McFadden pseudo-R²: {mcfadden:.3f} | deviance pseudo-R²: {dev_r2:.3f}")

d = cells[cells["E"] > 0].copy()
d["bin"] = pd.qcut(np.log10(d["E"]), 20, labels=False)
g = d.groupby("bin")["A"]
bm, bv, bn = g.mean(), g.var(ddof=1), g.size()
between       = np.average((bm - d["A"].mean())**2, weights=bn)
within_excess = np.average(np.maximum(bv - bm, 0), weights=bn)
share_raw     = between / (between + within_excess)

d["sA"] = np.sqrt(d["A"])
gs = d.groupby("bin")["sA"]
bms, bvs, bns = gs.mean(), gs.var(ddof=1), gs.size()
between_s       = np.average((bms - d["sA"].mean())**2, weights=bns)
within_excess_s = np.average(np.maximum(bvs - 0.25, 0), weights=bns)
share_sqrt      = between_s / (between_s + within_excess_s)
print(f"structural share explained by E: raw {share_raw:.1%} | √A robust {share_sqrt:.1%}")


In [ ]:
# ================= Panel a: AIS shipping activity density (two versions, dpi=500) =================
def draw_a(annot):
    E_map = np.where(domain, E_annual_full, np.nan)
    vals = E_map[np.isfinite(E_map)]
    bounds = np.unique(np.nanpercentile(vals, [10, 30, 50, 70, 80, 90, 95, 99.5]))
    ncol = len(bounds) - 1
    try:
        import cmocean; base = cmocean.cm.matter
    except ImportError:
        base = plt.cm.YlOrRd
    cmap = ListedColormap(base(np.linspace(0.15, 1.0, ncol)))
    cmap.set_under("white"); cmap.set_bad("white")
    fig = plt.figure(figsize=(13, 6.5))
    ax = fig.add_subplot(projection=ccrs.Robinson()); ax.set_global(); ax.set_facecolor("white")
    im = ax.imshow(E_map, origin="lower", extent=EXT, transform=ccrs.PlateCarree(),
                   cmap=cmap, norm=BoundaryNorm(bounds, ncol), interpolation="nearest", zorder=1)
    ax.add_feature(cfeature.LAND, facecolor="#eeeeee", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#888888", zorder=3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#b0b0b0", zorder=3)
    ax.gridlines(linewidth=0.3, color="#dddddd")
    cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02, ticks=bounds, spacing="uniform")
    cb.ax.set_yticklabels([f"{b:.0e}" for b in bounds])
    if annot:
        ax.set_title(f"a  Annual AIS traffic density per 1° cell   (domain cells: {int(domain.sum())})")
        cb.set_label("AIS position reports / year")
    return fig

save_two(draw_a, "fig1a_traffic_density")


In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

# ── Visual constants shared with the region-boundary figure ──────────────
GREY_LAND, INK, INK2 = '#dcdee0', '#1a1a1a', '#5a5f66'

# ================= Panel a: AIS shipping activity density (two versions, dpi=500)=================
def draw_a(annot):
    E_map  = np.where(domain, E_annual_full, np.nan)
    vals   = E_map[np.isfinite(E_map)]
    bounds = np.unique(np.nanpercentile(vals, [10, 30, 50, 70, 80, 90, 95, 99.5]))
    bounds = np.unique(np.round(bounds).astype(np.int64))      # round breakpoints so labels are exact values
    ncol   = len(bounds) - 1
    try:
        import cmocean; base = cmocean.cm.matter
    except ImportError:
        base = plt.cm.Blues
    cmap = ListedColormap(base(np.linspace(0.12, 0.97, ncol)))
    cmap.set_under('white'); cmap.set_bad('white')

    fig = plt.figure(figsize=(13.6, 7.6), dpi=500)
    ax  = fig.add_axes([0.045, 0.145, 0.915, 0.800],
                       projection=ccrs.Robinson(central_longitude=10))
    ax.set_global(); ax.set_facecolor('white')

    # graticule below data: visible only over empty ocean, does not cut shipping lanes
    ax.gridlines(lw=.4, color='#c9ccd0', alpha=.55, zorder=1)
    im = ax.imshow(E_map, origin='lower', extent=EXT, transform=ccrs.PlateCarree(),
                   cmap=cmap, norm=BoundaryNorm(bounds, ncol),
                   interpolation='nearest', zorder=2)
    ax.add_feature(cfeature.LAND,      facecolor=GREY_LAND, edgecolor='none', zorder=3)
    ax.add_feature(cfeature.COASTLINE, lw=.45, edgecolor='#ffffff', zorder=4)
    ax.add_feature(cfeature.BORDERS,   lw=.35, edgecolor='#ffffff', zorder=4)
    ax.spines['geo'].set_visible(False)

    # ── Thin bottom colorbar: equal-width blocks + white separators, integer ticks ──
    cax = fig.add_axes([0.270, 0.108, 0.460, 0.019])
    cb  = fig.colorbar(im, cax=cax, orientation='horizontal',
                       spacing='uniform', ticks=bounds)
    cb.outline.set_visible(False)
    if cb.dividers is not None:
        cb.dividers.set_color('white'); cb.dividers.set_linewidth(.8)
    cb.ax.tick_params(length=0, pad=3, labelsize=7.0, colors=INK2)
    cb.set_ticklabels([f'{int(b):,}' for b in bounds])
    for t in cb.ax.get_xticklabels():                 # pull in first/last labels to stay inside the frame
        t.set_ha('center')
    cb.ax.get_xticklabels()[0].set_ha('left')
    cb.ax.get_xticklabels()[-1].set_ha('right')

    if annot:
        fig.text(0.028, 0.962, 'a', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.055, 0.062,
                 'Annual AIS position reports per 1° cell over the analysis domain '
                 f'({int(domain.sum()):,} cells). Colour breaks follow\nwithin-domain percentiles '
                 '(10th–99.5th), so each class holds a comparable share of cells and the busiest '
                 'lanes stay separable;\nwhite marks cells outside the domain.',
                 fontsize=8.0, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_a, "fig1a_traffic_density")


In [ ]:
# ================= Panel b: accident counts (two versions, dpi=500) =================
def draw_b(annot):
    A_show = np.where(domain & (A_grid > 0), A_grid, np.nan)
    amax = int(np.nanmax(A_show))
    raw = [1, 2, 3, 5, 8, 13, 21, 34, 55]
    bounds_b = np.array([b for b in raw if b <= amax] + [amax + 1]); ncol_b = len(bounds_b) - 1
    try:
        import cmocean; base_b = cmocean.cm.amp
    except ImportError:
        base_b = plt.cm.Reds
    cmap_b = ListedColormap(base_b(np.linspace(0.30, 1.0, ncol_b)))
    cmap_b.set_under("white"); cmap_b.set_bad("white")
    fig = plt.figure(figsize=(13, 6.5))
    ax = fig.add_subplot(projection=ccrs.Robinson()); ax.set_global(); ax.set_facecolor("white")
    im = ax.imshow(A_show, origin="lower", extent=EXT, transform=ccrs.PlateCarree(),
                   cmap=cmap_b, norm=BoundaryNorm(bounds_b, ncol_b), interpolation="nearest", zorder=1)
    ax.add_feature(cfeature.LAND, facecolor="#eeeeee", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="#888888", zorder=3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="#b0b0b0", zorder=3)
    ax.gridlines(linewidth=0.3, color="#dddddd")
    cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02, ticks=bounds_b, spacing="uniform")
    if annot:
        n_acc_cells = int((A_grid[domain] > 0).sum())
        ax.set_title(f"b  Accident count per 1° cell   (accident cells: {n_acc_cells})")
        cb.set_label("accident count")
    return fig

save_two(draw_b, "fig1b_accidents")


In [ ]:
# ================= Panel b: accident counts (two versions, dpi=500)=================
def draw_b(annot):
    A_show = np.where(domain & (A_grid > 0), A_grid, np.nan)
    amax   = int(np.nanmax(A_show))
    raw    = [1, 5, 8, 15, 30, 50, 100,200,500]
    bounds_b = np.array([b for b in raw if b <= amax] + [amax + 1])
    ncol_b   = len(bounds_b) - 1
    try:
        import cmocean; base_b = cmocean.cm.amp
    except ImportError:
        base_b = plt.cm.Reds
    cmap_b = ListedColormap(base_b(np.linspace(0.30, 1.0, ncol_b)))
    cmap_b.set_under('white'); cmap_b.set_bad('white')

    fig = plt.figure(figsize=(13.6, 7.6), dpi=500)
    ax  = fig.add_axes([0.045, 0.145, 0.915, 0.800],
                       projection=ccrs.Robinson(central_longitude=10))
    ax.set_global(); ax.set_facecolor('white')

    ax.gridlines(lw=.4, color='#c9ccd0', alpha=.55, zorder=1)
    im = ax.imshow(A_show, origin='lower', extent=EXT, transform=ccrs.PlateCarree(),
                   cmap=cmap_b, norm=BoundaryNorm(bounds_b, ncol_b),
                   interpolation='nearest', zorder=2)
    ax.add_feature(cfeature.LAND,      facecolor=GREY_LAND, edgecolor='none', zorder=3)
    ax.add_feature(cfeature.COASTLINE, lw=.45, edgecolor='#ffffff', zorder=4)
    ax.add_feature(cfeature.BORDERS,   lw=.35, edgecolor='#ffffff', zorder=4)
    ax.spines['geo'].set_visible(False)

    # ── Thin bottom colorbar: equal-width blocks + white separators, integer ticks ──
    cax = fig.add_axes([0.270, 0.108, 0.460, 0.019])
    cb  = fig.colorbar(im, cax=cax, orientation='horizontal',
                       spacing='uniform', ticks=bounds_b)
    cb.outline.set_visible(False)
    if cb.dividers is not None:
        cb.dividers.set_color('white'); cb.dividers.set_linewidth(.8)
    cb.ax.tick_params(length=0, pad=3, labelsize=7.0, colors=INK2)
    lab = [f'{int(b)}' for b in bounds_b]
    lab[-1] = f'{amax}'                       # top class is [.., amax]; right end labeled with the true max
    cb.set_ticklabels(lab)
    cb.ax.get_xticklabels()[0].set_ha('left')
    cb.ax.get_xticklabels()[-1].set_ha('right')

    if annot:
        n_acc_cells = int((A_grid[domain] > 0).sum())
        fig.text(0.028, 0.962, 'b', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.055, 0.062,
                 f'Cargo-ship accident counts per 1° cell over the analysis domain '
                 f'({n_acc_cells:,} cells with at least one accident). Class breaks are\n'
                 'integer counts on a Fibonacci-like scale, so low-count cells stay distinguishable '
                 'while the few high-count cells do not\ncompress the ramp; white marks cells with no '
                 'recorded accident or outside the domain.',
                 fontsize=8.0, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_b, "fig1b_accidents")


In [ ]:
# ================= Panel c: scaling (two versions, dpi=500) =================
rate = df["A"].sum() / df["E"].sum()
df["mu0"] = rate * df["E"]
df["oe"]  = df["A"] / df["mu0"]
rho, p_sp = spearmanr(df["E"], df["A"])
p_txt = "p < 0.001" if p_sp < 1e-3 else f"p = {p_sp:.2g}"

def draw_c(annot):
    pos = df[df["A"] > 0]
    fig, ax = plt.subplots(figsize=(8, 6.2))
    sc = ax.scatter(pos["E"], pos["A"], c=pos["oe"].clip(0.1, 10), s=11, alpha=0.75,
                    cmap="RdBu_r", norm=LogNorm(vmin=0.1, vmax=10), edgecolors="none")
    xs = np.logspace(np.log10(df["E"].min()), np.log10(df["E"].max()), 200)
    mu = np.exp(a0) * xs ** beta
    m = mu >= 0.3
    ax.plot(xs[m], mu[m], "k-", lw=2, label=f"PPML  β={beta:.2f}")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(df["E"].min() * 0.8, df["E"].max() * 1.3)
    cb = fig.colorbar(sc, ax=ax, extend="both", pad=0.02)
    if annot:
        ax.set_xlabel("Annual AIS position reports per cell")
        ax.set_ylabel("Total accidents per cell")
        ax.set_title("c  scaling: accidents vs exposure")
        ax.text(0.03, 0.97,
                f"Spearman ρ = {rho:.3f}  ({p_txt})\n"
                f"PPML β = {beta:.3f}  (p$_{{β=1}}$={p_beta:.2g})\n"
                f"pseudo-R² = {mcfadden:.3f}  (predictive fit)\n"
                f"structural variance explained = {share_sqrt:.0%}\n"
                f"n = {len(df):,} cells",
                transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(boxstyle="round", fc="white", ec="#cccccc", alpha=.9))
        ax.legend(frameon=False, loc="lower right", fontsize=9)
        cb.set_label("O/E ratio")
    return fig

save_two(draw_c, "fig1c_scaling")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel c: scaling relation (scatter + O/E colors + PPML; colors and clipping as before)
#  Panel e: variance decomposition (structural / Poisson / unexplained), highlighting structural variance explained
#  Requires: df['A','E'], a0, beta, p_beta, mcfadden, save_two, FIGDIR
#        Optional: se_beta (if present, panel c shows the 95% CI of β)
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import spearmanr

INK, INK2, GREY = '#1a1a1a', '#5a5f66', '#9aa0a6'
C_STR, C_POI, C_RES = '#2f6fae', '#c3c8cd', '#c4614f'      # structural / Poisson / unexplained

def _g(name, default=None):
    return globals().get(name, default)

# ══════════════ 1. Statistics setup ══════════════
rate      = df['A'].sum() / df['E'].sum()
df['mu0'] = rate * df['E']                                 # β=1 null model (same overall rate)
df['oe']  = df['A'] / df['mu0']
rho, p_sp = spearmanr(df['E'], df['A'])
p_txt     = 'p < 0.001' if p_sp < 1e-3 else f'p = {p_sp:.2g}'

E  = df['E'].values.astype(float)
A  = df['A'].values.astype(float)
MU = np.exp(a0) * E ** beta                                # PPML fitted expectation

# ══════════════ 2. Variance decomposition (sqrt scale: Poisson variance stabilized)══════════════
# v_tot = Var[sqrt(A)]                            total observed variance
# v_str = Var[sqrt(mu_hat)]                       structural variance explained by exposure
# v_sp  = E{ Var[sqrt(Poisson(mu_hat))] }         structural + Poisson (Monte Carlo, incl. small counts and zeros)
# v_res = v_tot - v_str - v_poi                   unexplained variance beyond Poisson (overdispersion)
rng   = np.random.default_rng(0)
v_tot = float(np.var(np.sqrt(A),  ddof=1))
v_str = float(np.var(np.sqrt(MU), ddof=1))
v_sp  = float(np.mean([np.var(np.sqrt(rng.poisson(MU)), ddof=1) for _ in range(400)]))
v_poi = max(0.0, v_sp - v_str)
v_res = v_tot - v_str - v_poi

s_str, s_poi, s_res = v_str / v_tot, v_poi / v_tot, v_res / v_tot
if s_res < 0:                                              # underdispersion: data smoother than Poisson
    print(f'⚠ unexplained share is negative ({s_res:+.1%}) -> data underdispersed; segment shown as 0 in bar')
    _sc = s_str + s_poi
    s_str, s_poi, s_res = s_str / _sc, s_poi / _sc, 0.0

print(f'Variance decomposition (sqrt scale, n = {len(df):,} cells)')
print(f'  structural (exposure)  {s_str:6.1%}')
print(f'  Poisson noise          {s_poi:6.1%}')
print(f'  unexplained            {s_res:6.1%}')
if _g('share_sqrt') is not None:
    print(f'  (original share_sqrt = {share_sqrt:.1%}, diff {abs(share_sqrt - s_str):.1%})')
pd.DataFrame([dict(structural=s_str, poisson=s_poi, unexplained=s_res, beta=beta,
                   se_beta=_g('se_beta'), p_beta=p_beta, pseudo_R2=_g('mcfadden'),
                   spearman_rho=rho, n_cells=len(df))]
             ).to_csv(os.path.join(FIGDIR, 'fig1_scaling_stats.csv'), index=False)

# ══════════════ 3. Panel c: scaling relation ══════════════
def draw_c(annot):
    pos = df[df['A'] > 0]
    fig = plt.figure(figsize=(7.8, 6.4), dpi=500)
    ax  = fig.add_axes([0.125, 0.135, 0.745, 0.775])

    sc = ax.scatter(pos['E'], pos['A'], c=pos['oe'].clip(0.1, 10), s=11, alpha=0.75,
                    cmap='RdBu_r', norm=LogNorm(vmin=0.1, vmax=10), edgecolors='none',
                    zorder=2)

    xs = np.logspace(np.log10(df['E'].min()), np.log10(df['E'].max()), 200)
    ax.plot(xs, rate * xs, ls=(0, (5, 3)), lw=1.4, color=GREY, zorder=3,
            label='proportional to exposure  (β = 1)')
    mu = np.exp(a0) * xs ** beta
    m  = mu >= 0.3                                          # no extrapolation below 0.3 expected accidents per cell
    ax.plot(xs[m], mu[m], 'k-', lw=2, zorder=4, label=f'PPML  β={beta:.2f}')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(df['E'].min() * 0.8, df['E'].max() * 1.3)
    ax.set_ylim(1e-1, df['A'].max() * 1.6)        # y-axis from 10⁻¹, headroom at top
    ax.minorticks_off()
    ax.tick_params(which='major', length=3.2, width=.6, direction='out',
                   labelsize=8.4, color=INK2, labelcolor=INK2)
    ax.tick_params(which='minor', length=1.8, width=.5, direction='out', color='#b8bdc2')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color('#c9ccd0'); ax.spines[s].set_linewidth(.8)
    ax.grid(which='major', lw=.4, color='#e7e9eb', zorder=0)

    cax = fig.add_axes([0.888, 0.135, 0.018, 0.400])
    cb  = fig.colorbar(sc, cax=cax, extend='both')
    cb.outline.set_visible(False)
    cb.ax.tick_params(which='both', length=2.6, width=.5, direction='out',
                      labelsize=7.4, color=INK2, labelcolor=INK2)

    if annot:
        fig.text(0.022, 0.955, 'c', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.085, 0.958, 'Accidents scale sub-linearly with exposure',
                 fontsize=12, fontweight='bold', color=INK)
        ax.set_xlabel('Annual AIS position reports per cell', fontsize=9.5, color=INK)
        ax.set_ylabel('Total accidents per cell', fontsize=9.5, color=INK)
        cb.set_label('O/E ratio', fontsize=8.4, color=INK2)
        ci = (f'  [{beta - 1.96 * se_beta:.3f}, {beta + 1.96 * se_beta:.3f}]'
              if _g('se_beta') is not None else '')
        ax.text(.035, .965,
                f'β = {beta:.3f}{ci}\n'
                f'p(β = 1) = {p_beta:.2g}\n'
                f'Spearman ρ = {rho:.3f}  ({p_txt})\n'
                f'n = {len(df):,} cells',
                transform=ax.transAxes, va='top', fontsize=8.6, color=INK,
                linespacing=1.55,
                bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#dcdee0', lw=.7))
        ax.legend(frameon=False, loc='lower right', fontsize=8.4, labelcolor=INK2)
        fig.text(0.125, 0.048,
                 f'Each point is a 1° cell; colour gives the ratio of observed to exposure-implied '
                 f'accidents (O/E, clipped to 0.1–10). Only the {len(pos):,} cells with at least\n'
                 f'one accident can appear on log axes, but the PPML fit uses all {len(df):,} cells '
                 'including zeros; the fitted line is drawn where the expectation exceeds\n'
                 '0.3 accidents per cell. The dashed line is the null in which accidents are '
                 'proportional to exposure.',
                 fontsize=7.8, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_c, 'fig1c_scaling')

# ══════════════ 4. Panel e: variance decomposition ══════════════
def draw_e(annot):
    parts = [('Exposure (structural)',        s_str, C_STR),
             ('Poisson sampling noise',       s_poi, C_POI),
             ('Unexplained (overdispersion)', s_res, C_RES)]

    fig = plt.figure(figsize=(9.4, 3.3), dpi=500)
    ax  = fig.add_axes([0.065, 0.455, 0.895, 0.185])
    left = 0.0
    for name, v, col in parts:
        if v <= 0:
            continue
        ax.barh(0, v, left=left, height=1.0, color=col, edgecolor='white', lw=2.2, zorder=2)
        if v > .07:                                         # write percentage inside segment only if wide enough
            ax.text(left + v / 2, 0, f'{v:.0%}', ha='center', va='center',
                    fontsize=10.5, fontweight='bold', zorder=3,
                    color='white' if col != C_POI else INK)
        left += v

    ax.set_xlim(0, 1); ax.set_ylim(-.5, .5); ax.set_yticks([])
    ax.set_xticks(np.arange(0, 1.01, .25))
    ax.set_xticklabels([f'{int(t * 100)}' for t in np.arange(0, 1.01, .25)])
    ax.set_xticks(np.arange(0, 1.01, .05), minor=True)
    ax.tick_params(which='major', length=3.2, width=.6, direction='out',
                   pad=3, labelsize=8, color=INK2, labelcolor=INK2)
    ax.tick_params(which='minor', length=1.8, width=.5, direction='out', color='#b8bdc2')
    for s in ax.spines.values():
        s.set_visible(False)

    if annot:
        fig.text(0.022, 0.905, 'e', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.078, 0.910, 'Exposure explains most of the structural variance',
                 fontsize=12, fontweight='bold', color=INK)
        x = 0.065
        for name, v, col in parts:                          # legend: color patch + name + value
            fig.patches.append(plt.Rectangle((x, 0.300), .014, .052, transform=fig.transFigure,
                                             facecolor=col, edgecolor='none', zorder=5))
            fig.text(x + .022, 0.326, name, fontsize=8.8, fontweight='bold',
                     color=INK, va='center')
            fig.text(x + .022, 0.272, f'{v:.1%}', fontsize=8.4, fontweight='bold',
                     color=col, va='center')
            x += .032 + len(name) * .0062
        fig.text(0.962, 0.660, 'share of Var[√A]  (%)', ha='right', fontsize=8,
                 color=INK2, va='bottom')
        fig.text(0.065, 0.180,
                 'Cell-level variance in accident counts, decomposed on the square-root scale '
                 'where Poisson variance is stabilised. The structural term is the\nvariance of the '
                 f'PPML-fitted expectation (β = {beta:.3f}); the Poisson term is the variance a '
                 'perfectly specified model would still produce, estimated by\nsimulating counts '
                 'from the fitted expectations. Predictive fit is modest by construction '
                 f'(pseudo-R² = {_g("mcfadden", float("nan")):.3f}) because most\nremaining '
                 'variance is irreducible sampling noise, not model error.',
                 fontsize=7.8, color=INK2, va='top', linespacing=1.55)
    return fig

save_two(draw_e, 'fig1e_variance')


In [ ]:
# Verify the identity above and give residual-form R² a "perfect-model ceiling"
r = np.sqrt(A) - np.sqrt(MU)
T = np.var(np.sqrt(A), ddof=1)
P = np.var(np.sqrt(MU), ddof=1)
R = np.var(r, ddof=1)
C = np.cov(np.sqrt(MU), r, ddof=1)[0, 1]
print(f'generative  Var(√μ̂)/Var(√A)    = {P / T:.1%}')
print(f'residual    1 − Var(r)/Var(√A) = {1 - R / T:.1%}')
print(f'difference  −2·Cov/Var(√A)     = {-2 * C / T:.1%}')
print(f'share of zero-accident cells   = {(A == 0).mean():.1%}')

R2r = lambda a: 1 - np.var(np.sqrt(a) - np.sqrt(MU), ddof=1) / np.var(np.sqrt(a), ddof=1)
sim = np.array([R2r(rng.poisson(MU)) for _ in range(400)])
print(f'\nresidual-form R²: observed {R2r(A):.1%}   perfect-model ceiling {sim.mean():.1%} ± {sim.std():.1%}')

# Also report Monte Carlo error of the variance decomposition, to confirm the two 15.4% are not a numerical coincidence
sv = np.array([np.var(np.sqrt(rng.poisson(MU)), ddof=1) for _ in range(400)])
print(f'v_sp = {sv.mean():.4f} ± {sv.std() / np.sqrt(len(sv)):.4f}  (MC s.e.)')


In [ ]:
import statsmodels.api as sm
from scipy.stats import gamma, norm

mu = MU                                    # PPML fitted expectation
ok = np.isfinite(mu) & (mu > 0) & np.isfinite(A)
a_, m_ = A[ok], mu[ok]
print(f'valid sample {ok.sum():,} / {len(A):,}')

# 1) Cameron–Trivedi auxiliary regression (also an overdispersion test)
#    For NB2: E[((A−μ)² − A)/μ] = α·μ, regression without intercept
z    = ((a_ - m_) ** 2 - a_) / m_
aux  = sm.OLS(z, m_[:, None]).fit(cov_type='HC3')
a_ct, t_ct = aux.params[0], aux.tvalues[0]
print(f'α (C&T auxiliary reg.) = {a_ct:.4f}   t = {t_ct:.1f}   p = {2 * norm.sf(abs(t_ct)):.1e}')

# 2) Method of moments (closed form, most stable)
a_mm = np.sum((a_ - m_) ** 2 - m_) / np.sum(m_ ** 2)
print(f'α (moments)              = {a_mm:.4f}')

# 3) Profile likelihood: GLM with fixed α over a grid, point estimate and 95% CI
X    = sm.add_constant(np.log(m_))         # log μ̂ as single covariate, equivalent to a linear reparametrization of log E
grid = np.logspace(-2, 2, 121)
llf  = np.array([sm.GLM(a_, X, family=sm.families.NegativeBinomial(alpha=g)).fit().llf
                 for g in grid])
a_pl = grid[np.argmax(llf)]
inci = grid[llf >= llf.max() - 1.92]       # likelihood drop 1.92 -> 95% CI
print(f'α (profile lik.)         = {a_pl:.4f}   95% CI [{inci.min():.4f}, {inci.max():.4f}]')

# ── Readout ──
alpha = a_pl
k, sc = 1 / alpha, alpha
print(f'\nP90/P10 of η at equal traffic = {gamma.ppf(.9, k, scale=sc) / gamma.ppf(.1, k, scale=sc):.1f}×'
      f'   residual CV = {np.sqrt(alpha):.2f}')
V_mu  = m_.var(ddof=1)
share = V_mu / (V_mu * (1 + alpha) + alpha * m_.mean() ** 2)
print(f'systematic variance share explained by traffic = {share:.1%}   remaining {1 - share:.1%}')

# ── Is NB2 adequate: zero-share comparison ──
p0_nb = np.mean((1 / (1 + alpha * m_)) ** (1 / alpha))
print(f'\nzero-cell share: observed {(a_ == 0).mean():.1%}   NB2 {p0_nb:.1%}   Poisson {np.exp(-m_).mean():.1%}')


In [ ]:
# 1) Spatial concentration of accidents: directly verifiable, no distributional assumption
s = np.sort(A)[::-1]
for p in (0.005, 0.01, 0.05, 0.10):
    k = max(1, int(round(p * len(s))))
    print(f'top {p:.1%} cells by accidents ({k} cells) hold {s[:k].sum() / s.sum():.1%} of all accidents')

# 2) Share of cells above mean residual risk (gamma probability, stable for large α)
from scipy.stats import gamma
for a in (7.69, 10.80):
    print(f'α={a:.2f}:  P(η > 1) = {gamma.sf(1, 1 / a, scale=a):.1%}'
          f'   CV(η) = {np.sqrt(a):.2f}')

# 3) Fine profile-likelihood scan for a proper CI
import statsmodels.api as sm
X    = sm.add_constant(np.log(MU))
grid = np.linspace(5, 16, 221)
llf  = np.array([sm.GLM(A, X, family=sm.families.NegativeBinomial(alpha=g)).fit().llf
                 for g in grid])
inci = grid[llf >= llf.max() - 1.92]
print(f'α = {grid[np.argmax(llf)]:.2f}   95% CI [{inci.min():.2f}, {inci.max():.2f}]')

# 4) Robustness: drop the top 0.1% cells by accidents, how much α falls
cut = np.quantile(A, 0.999)
m   = A <= cut
z   = ((A[m] - MU[m]) ** 2 - A[m]) / MU[m]
print(f'α after dropping top 0.1% cells (moments) = {np.sum((A[m] - MU[m]) ** 2 - MU[m]) / np.sum(MU[m] ** 2):.2f}')

# 5) Nonparametric: within-decile overdispersion ratio by traffic decile (snippet given earlier)


In [ ]:
# ══ Concentration of traffic vs accidents ══
Av, Ev = A.astype(float), E.astype(float)
n      = len(Av)
PS     = (0.005, 0.01, 0.05, 0.10)

def cum_share(rank_by, val, p):
    """Take the top-p share of cells by rank_by (descending); return their share of total val"""
    k = max(1, int(round(p * n)))
    idx = np.argsort(-rank_by)[:k]
    return val[idx].sum() / val.sum(), k

print(f'{"":>8s} {"cells":>6s} {"traffic":>8s} {"accid.":>8s} {"group O/E":>9s}')
print('— sorted by traffic —')
for p in PS:
    sE, k = cum_share(Ev, Ev, p)
    sA, _ = cum_share(Ev, Av, p)
    print(f'top {p:>5.1%} {k:>6d} {sE:>8.1%} {sA:>8.1%} {sA / sE:>9.2f}')
print('— sorted by accidents —')
for p in PS:
    sA, k = cum_share(Av, Av, p)
    sE, _ = cum_share(Av, Ev, p)
    print(f'top {p:>5.1%} {k:>6d} {sE:>8.1%} {sA:>8.1%} {sA / sE:>9.2f}')

# Overlap of the two top sets: are accident hotspots the traffic hotspots?
print('\nOverlap (both top-k)')
for p in PS:
    k = max(1, int(round(p * n)))
    sa = set(np.argsort(-Av)[:k]); se = set(np.argsort(-Ev)[:k])
    print(f'top {p:>5.1%}  k={k:>5d}   overlap {len(sa & se):>5d} cells = {len(sa & se) / k:.1%}')

# Gini: 0 = perfectly even, 1 = all in one cell
gini = lambda v: (2 * np.sum((np.arange(1, n + 1)) * np.sort(v)) / (n * v.sum()) - (n + 1) / n)
print(f'\nGini: traffic {gini(Ev):.3f}   accidents {gini(Av):.3f}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel f: spatial concentration of accidents vs shipping activity mismatch
#    top: locations of the 171 grid cells with most accidents
#    bottom: cumulative share curves (grid cells sorted by accidents, descending)
#  Requires: df['A','E'] + lat/lon columns, MU (else proportional null model), save_two
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs, cartopy.feature as cfeature

INK, INK2, GREY, GREY_LAND = '#1a1a1a', '#5a5f66', '#9aa0a6', '#dcdee0'
C_ACC, C_TRF = '#c4614f', '#2f6fae'
TOP_P = 0.005                                    # threshold defining "most concentrated"

def _col(d, cands, what):
    for c in cands:
        if c in d.columns:
            return c
    raise KeyError(f'{what} column not found; candidates {cands}, actual {list(d.columns)}')

LAT = _col(df, ['lat', 'lat_c', 'lat_center', 'cell_lat', 'y'], 'latitude')
LON = _col(df, ['lon', 'lon_c', 'lon_center', 'cell_lon', 'x'], 'longitude')

Av, Ev = df['A'].values.astype(float), df['E'].values.astype(float)
n      = len(Av)
K      = max(1, int(round(TOP_P * n)))

oA   = np.argsort(-Av)                           # sort by accidents, descending
cumA = np.cumsum(Av[oA]) / Av.sum()              # cumulative accident share
cumE = np.cumsum(Ev[oA]) / Ev.sum()              # cumulative traffic share, same ordering
oE   = np.argsort(-Ev)                           # sort by traffic, descending (reference)
cumR = np.cumsum(Ev[oE]) / Ev.sum()
xs   = np.arange(1, n + 1) / n

sA, sE = cumA[K - 1], cumE[K - 1]
print(f'top {TOP_P:.1%} ({K} cells): accidents {sA:.1%}  traffic {sE:.1%}  excess {sA / sE:.1f}×')

def draw_f(annot):
    fig = plt.figure(figsize=(11.2, 8.6), dpi=500)

    # ── Top: hotspot locations ──
    axm = fig.add_axes([0.055, 0.520, 0.890, 0.400],
                       projection=ccrs.Robinson(central_longitude=10))
    axm.set_global()
    axm.gridlines(lw=.4, color='#c9ccd0', alpha=.55, zorder=0)
    axm.add_feature(cfeature.LAND, facecolor=GREY_LAND, edgecolor='none', zorder=1)
    axm.add_feature(cfeature.COASTLINE, lw=.45, edgecolor='#ffffff', zorder=2)
    axm.spines['geo'].set_visible(False)
    top = df.iloc[oA[:K]]
    axm.scatter(top[LON], top[LAT], s=13, marker='s', c=C_ACC, alpha=.85, lw=0,
                transform=ccrs.PlateCarree(), zorder=3)

    # ── Bottom: cumulative share curves ──
    ax = fig.add_axes([0.085, 0.155, 0.855, 0.300])
    ax.fill_between(xs, cumE * 100, cumA * 100, where=cumA >= cumE,
                    color=C_ACC, alpha=.10, lw=0, zorder=1)
    ax.plot(xs, cumR * 100, ls=(0, (5, 3)), lw=1.3, color=GREY, zorder=2,
            label='traffic, cells ranked by traffic')
    ax.plot(xs, cumE * 100, '-', lw=1.8, color=C_TRF, zorder=3,
            label='traffic, cells ranked by accidents')
    ax.plot(xs, cumA * 100, '-', lw=2.2, color=C_ACC, zorder=4,
            label='accidents, cells ranked by accidents')

    ax.axvline(TOP_P, lw=.8, ls=':', color=INK2, zorder=2)
    for y, col in ((sA * 100, C_ACC), (sE * 100, C_TRF)):
        ax.plot(TOP_P, y, 'o', ms=5.5, mfc='white', mec=col, mew=1.6, zorder=6)

    ax.set_xscale('log')
    ax.set_xlim(0.8 / n, 1); ax.set_ylim(0, 102)
    tk = [1e-4, 1e-3, TOP_P, 1e-2, 5e-2, 1e-1, 1]
    ax.xaxis.set_major_locator(mticker.FixedLocator(tk))
    ax.xaxis.set_major_formatter(mticker.FixedFormatter(
        ['0.01%', '0.1%', '0.5%', '1%', '5%', '10%', '100%']))
    ax.minorticks_off()
    ax.set_yticks(np.arange(0, 101, 25))
    ax.tick_params(which='both', length=3.2, width=.6, direction='out',
                   labelsize=8.4, color=INK2, labelcolor=INK2)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color('#c9ccd0'); ax.spines[s].set_linewidth(.8)
    ax.grid(which='major', axis='y', lw=.4, color='#e7e9eb', zorder=0)

    if annot:
        fig.text(0.022, 0.958, 'f', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.078, 0.961, 'Accidents concentrate where traffic does not',
                 fontsize=12.5, fontweight='bold', color=INK)
        fig.text(0.055, 0.930,
                 f'The {K} highest-accident cells ({TOP_P:.1%} of the domain)',
                 fontsize=8.6, color=INK2)
        ax.set_xlabel('cells, ranked in descending order (share of domain, log scale)',
                      fontsize=9, color=INK)
        ax.set_ylabel('cumulative share (%)', fontsize=9, color=INK)
        ax.annotate(f'{sA:.1%} of accidents', xy=(TOP_P, sA * 100),
                    xytext=(TOP_P * 2.6, sA * 100 + 9), fontsize=8.8,
                    fontweight='bold', color=C_ACC,
                    arrowprops=dict(arrowstyle='-', lw=.7, color=C_ACC))
        ax.annotate(f'{sE:.1%} of traffic', xy=(TOP_P, sE * 100),
                    xytext=(TOP_P * 2.6, sE * 100 - 14), fontsize=8.8,
                    fontweight='bold', color=C_TRF,
                    arrowprops=dict(arrowstyle='-', lw=.7, color=C_TRF))
        ax.legend(frameon=False, loc='center right', fontsize=8.2, labelcolor=INK2)
        fig.text(0.085, 0.098,
                 f'Cells are ranked by accident count and cumulated from the highest. The {K} '
                 f'highest-accident cells carry {sA:.1%} of all accidents but only {sE:.1%} of '
                 f'traffic, a {sA / sE:.1f}-fold excess;\nthe dashed curve shows that traffic is '
                 'itself concentrated, but in different cells (Gini 0.863 for traffic against 0.974 '
                 'for accidents). Squares above mark those cells,\nwhich fall predominantly in '
                 'strait constrictions, port approaches and pilotage waters.',
                 fontsize=7.8, color=INK2, va='top', linespacing=1.55)
    return fig

save_two(draw_f, 'fig1f_concentration')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel f: spatial concentration of accidents vs shipping activity mismatch (cumulative share curves, no text)
#  Requires: df['A','E'], save_two
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

INK2, GREY = '#5a5f66', '#9aa0a6'
C_ACC, C_TRF = '#c4614f', '#2f6fae'
TOP_P = 0.005                                    # threshold defining "most concentrated"

Av, Ev = df['A'].values.astype(float), df['E'].values.astype(float)
n      = len(Av)
K      = max(1, int(round(TOP_P * n)))

oA   = np.argsort(-Av)                           # sort by accidents, descending
cumA = np.cumsum(Av[oA]) / Av.sum()              # cumulative accident share
cumE = np.cumsum(Ev[oA]) / Ev.sum()              # cumulative traffic share, same ordering
oE   = np.argsort(-Ev)                           # sort by traffic, descending (reference)
cumR = np.cumsum(Ev[oE]) / Ev.sum()
xs   = np.arange(1, n + 1) / n

sA, sE = cumA[K - 1], cumE[K - 1]
print(f'top {TOP_P:.1%} ({K} cells): accidents {sA:.1%}  traffic {sE:.1%}  excess {sA / sE:.1f}×')

def draw_f(annot):
    fig = plt.figure(figsize=(8, 10), dpi=500)
    ax  = fig.add_axes([0.085, 0.135, 0.885, 0.830])

    ax.fill_between(xs, cumE * 100, cumA * 100, where=cumA >= cumE,
                    color=C_ACC, alpha=.10, lw=0, zorder=1)
    ax.plot(xs, cumR * 100, ls=(0, (5, 3)), lw=1.3, color=GREY,  zorder=2)
    ax.plot(xs, cumE * 100, '-',            lw=1.8, color=C_TRF, zorder=3)
    ax.plot(xs, cumA * 100, '-',            lw=2.2, color=C_ACC, zorder=4)

    ax.axvline(TOP_P, lw=.8, ls=':', color=INK2, zorder=2)
    for y, col in ((sA * 100, C_ACC), (sE * 100, C_TRF)):
        ax.plot(TOP_P, y, 'o', ms=5.5, mfc='white', mec=col, mew=1.6, zorder=6)

    ax.set_xscale('log')
    ax.set_xlim(0.8 / n, 1); ax.set_ylim(0, 102)
    tk = [1e-4, 1e-3, TOP_P, 1e-2, 5e-2, 1e-1, 1]
    ax.xaxis.set_major_locator(mticker.FixedLocator(tk))
    ax.xaxis.set_major_formatter(mticker.FixedFormatter(
        ['0.01%', '0.1%', '0.5%', '1%', '5%', '10%', '100%']))
    ax.minorticks_off()
    ax.set_yticks(np.arange(0, 101, 25))
    ax.tick_params(which='both', length=3.2, width=.6, direction='out',
                   labelsize=8.4, color=INK2, labelcolor=INK2)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color('#c9ccd0'); ax.spines[s].set_linewidth(.8)
    ax.grid(which='major', axis='y', lw=.4, color='#e7e9eb', zorder=0)
    return fig

save_two(draw_f, 'fig1f_concentration')

In [ ]:
# ================= Panel d (main): seasonal covariation (two versions, dpi=500) =================
# Storm season definitions (fixed before data, external sources): ONDJFM = extended extratropical winter (storm-track convention);
# Jun-Nov = JMA NW Pacific typhoon climatology (~89% TC); Mediterranean = calm control under same rule
# Stats always use whole months; shading edges set explicitly in SEASONS, decoupled from stats
from scipy.stats import binomtest

mcand = [c for c in ["acc_month", "month", "casualty_month"] if c in acc.columns]
if mcand:
    month = pd.to_numeric(acc[mcand[0]], errors="coerce")
else:
    dcand = [c for c in ["acc_date", "casualty_date", "date", "Casualty Date"] if c in acc.columns]
    assert dcand, f"month/date column not found: {list(acc.columns)}"
    month = pd.to_datetime(acc[dcand[0]], errors="coerce").dt.month
laD = pd.to_numeric(acc["lat_dec"], errors="coerce")
loD = pd.to_numeric(acc["lon_dec"], errors="coerce")
yrD = pd.to_numeric(acc["acc_year"], errors="coerce")
base = laD.notna() & loD.notna() & month.notna() & yrD.between(YR_MIN, YR_MAX)

REGIONS = {
    "NW Europe (48-62N, 12W-10E)":   dict(lat=(48, 62), lon=(-12, 10),   col="#2166ac",
                                          peak=(10, 11, 12, 1, 2, 3)),
    "E/SE Asia (5-30N, 105-140E)":   dict(lat=(5, 30),  lon=(105, 140),  col="#b2182b",
                                          peak=(6, 7, 8, 9, 10, 11)),
    "N Pacific (40-60N, 140E-140W)": dict(lat=(40, 60), lon=(140, -140), col="#7b3294",
                                          peak=(10, 11, 12, 1, 2, 3)),
    "Mediterranean (calm control)":  dict(lat=(30, 45), lon=(-5, 36),    col="#4dac26",
                                          peak=()),
}

# ---- Shading: spans give plot intervals explicitly; change edges here only, stats unaffected ----
# Current definition: edges on month ticks (Oct tick->Mar tick / Jun tick->Nov tick)
# For a "whole-month cell edge" definition, use [(9.5, 12.5), (0.5, 3.5)] and [(5.5, 11.5)]
SEASONS = [
    dict(spans=[(10, 12.5), (0.5, 3)],
         style=dict(facecolor="#2166ac", alpha=0.10, lw=0),
         patch=dict(facecolor="#2166ac", alpha=0.25),
         label="shaded: extended winter, ONDJFM  (applies to NW Europe & N Pacific)"),
    dict(spans=[(6, 11)],
         style=dict(facecolor="none", edgecolor="#b2182b", hatch="///", lw=0, alpha=0.35),
         patch=dict(facecolor="none", edgecolor="#b2182b", hatch="///"),
         label="hatched: typhoon season, Jun-Nov  (applies to E/SE Asia)"),
]

# ---- Stats computed once: shares, season ratio, binomial test (whole months) ----
stats = {}
for name, R in REGIONS.items():
    (la0, la1), (lo0, lo1) = R["lat"], R["lon"]
    sel = loD.between(lo0, lo1) if lo0 < lo1 else ((loD >= lo0) | (loD <= lo1))
    mo = month[base & laD.between(la0, la1) & sel].astype(int)
    cnt = mo.value_counts().reindex(range(1, 13), fill_value=0)
    n = int(cnt.sum()); share = cnt / n * 100
    if R["peak"]:
        k = int(mo.isin(R["peak"]).sum()); L = len(R["peak"])
        ratio = (k / L) / ((n - k) / (12 - L))
        pval  = binomtest(k, n, L / 12).pvalue
    else:
        ratio, pval = np.nan, np.nan
    stats[name] = dict(n=n, share=share, ratio=ratio, p=pval)
    print(f"{name:34s} n={n:6,d}" + (f"  ratio={ratio:.2f}  p={pval:.2e}" if R["peak"] else "  (control)"))

pd.DataFrame({k: dict(n=v["n"], ratio=v["ratio"], p=v["p"]) for k, v in stats.items()}
             ).T.to_csv(os.path.join(FIGDIR, "fig1d_season_stats.csv"))   # raw data for SI table

MONTH_LAB = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def draw_d(annot):
    fig, ax = plt.subplots(figsize=(9, 5.4))
    line_h, shade_h = [], []
    for s in SEASONS:                                   # full-height background; overlap = blue fill + red hatch
        for x0, x1 in s["spans"]:
            ax.axvspan(x0, x1, zorder=0, **s["style"])
        shade_h.append(Patch(**s["patch"], label=s["label"]))
    for name, R in REGIONS.items():
        st = stats[name]; col = R["col"]; ls = "-" if R["peak"] else "--"
        ax.plot(range(1, 13), st["share"], color=col, marker="o", ms=4, lw=1.8, ls=ls, zorder=3)
        lab = f"{name}  n={st['n']:,}"
        if R["peak"]:
            lab += f"  (×{st['ratio']:.2f} in season)"
        line_h.append(Line2D([0], [0], color=col, marker="o", ms=4, lw=1.8, ls=ls, label=lab))
    ax.axhline(100 / 12, color="#999999", lw=0.8, ls=":", zorder=2)
    line_h.append(Line2D([0], [0], color="#999999", lw=0.8, ls=":", label="uniform (8.3%)"))
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LAB, fontsize=8)
    ax.set_xlim(0.5, 12.5)                              # keep: axis ends = Jan left / Dec right edge so shading aligns
    ax.set_ylim(bottom=min(3.5, ax.get_ylim()[0]))
    if annot:
        ax.set_xlabel("month"); ax.set_ylabel("share of region's accidents (%)")
        ax.set_title("d  seasonality of accidents by region (2002-2021)")
        ax.legend(handles=line_h + shade_h, frameon=False, fontsize=7.5,
                  loc="lower center", ncol=2)
    ax.grid(alpha=0.25, lw=0.4)
    return fig

save_two(draw_d, "fig1d_seasonality")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel d: region x month accident-share heatmap
#  Color = deviation from uniform month (pp, diverging colormap, white = 8.3%)
#  Number = month's share of the region's accidents (%), rows sum to 100%
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

INK, INK2 = '#1a1a1a', '#5a5f66'
MONTH_LAB = list('JFMAMJJASOND')
NORM_BY   = 'row'      # 'row' = share of region total (rows sum 100%); 'all' = share of grand total (table sums 100%)
SHOW_VAL  = True       # annotate values in cells

# ══════════════ 1. Region boundaries (verbatim from M2_merge CELL 1)══════════════
# Structure: region -> subregion -> [lat/lon rectangles...], written (S, N, W, E) = (south, north, west, east)
# Rule: lat ∈ [S, N) and lon ∈ [W, E), always half-open; multiple rectangles of a subregion are unioned
REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic': [(36, 45, -15, -6)],
        'Biscay_Celtic':   [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':       [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}
ROW_META = [('NW_Europe',      'NW Europe'),
            ('NW_Atlantic',    'NW Atlantic'),
            ('Asia',           'E & SE Asia'),
            ('Med_Black',      'Mediterranean–Black Sea'),
            ('Baltic',         'Baltic Sea'),
            ('Indian_Ocean_N', 'N Indian Ocean')]
keys = [k for k, _ in ROW_META]
assert set(keys) == set(REGIONS), 'row order inconsistent with region definitions'

def region_of(lat, lon):
    """Vectorized region assignment. Returns (region, subregion) object arrays; None outside.
       A point in multiple boxes goes to the first hit in REGIONS order, with a warning."""
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    reg    = np.full(lat.shape, None, object)
    sub    = np.full(lat.shape, None, object)
    filled = np.zeros(lat.shape, bool)
    nhit   = np.zeros(lat.shape, int)
    for r, subs in REGIONS.items():
        for s, boxes in subs.items():
            hit = np.zeros(lat.shape, bool)
            for (S, N, W, E) in boxes:
                hit |= (lat >= S) & (lat < N) & (lon >= W) & (lon < E)
            nhit += hit
            take = hit & ~filled
            reg[take], sub[take] = r, s
            filled |= take
    n_ov = int((nhit > 1).sum())
    if n_ov:
        print(f'⚠ {n_ov} points fall in multiple boxes (assigned by definition order); check for overlapping boundaries')
    return reg, sub

# Print region boundaries for checking and the methods section
print('Region boundaries (lat ∈ [S,N), lon ∈ [W,E))')
for r, subs in REGIONS.items():
    print(f'  {r}')
    for s, boxes in subs.items():
        bt = '  ∪  '.join(f'{S}–{N}N, {W}–{E}E' for S, N, W, E in boxes)
        print(f'    {s:<16s} {bt}')

# ══════════════ 2. Accident region assignment and monthly stats ══════════════
_mcol = next((c for c in ['acc_month', 'month', 'casualty_month'] if c in acc.columns), None)
if _mcol:
    _mo = pd.to_numeric(acc[_mcol], errors='coerce')
else:
    _dcol = next(c for c in ['acc_date', 'casualty_date', 'date'] if c in acc.columns)
    _mo = pd.to_datetime(acc[_dcol], errors='coerce').dt.month

D = pd.DataFrame({
    'lat':   pd.to_numeric(acc['lat_dec'],  errors='coerce'),
    'lon':   pd.to_numeric(acc['lon_dec'],  errors='coerce'),
    'month': _mo,
    'year':  pd.to_numeric(acc['acc_year'], errors='coerce'),
}).dropna(subset=['lat', 'lon', 'month', 'year'])
D = D[D['year'].between(YR_MIN, YR_MAX)].copy()
D['region'], D['subregion'] = region_of(D['lat'].values, D['lon'].values)

n_in, n_out = int(D['region'].notna().sum()), int(D['region'].isna().sum())
D = D.dropna(subset=['region'])
YR0, YR1 = int(D['year'].min()), int(D['year'].max())

O = (pd.crosstab(D['region'], D['month'].astype(int))
       .reindex(keys).reindex(columns=range(1, 13)).fillna(0.0))
S    = O.div(O.sum(axis=1), axis=0) * 100 if NORM_BY == 'row' else O / O.values.sum() * 100
n_r  = O.sum(axis=1).astype(int)
UNIF = 100 / 12

print(f'\naccidents: in regions {n_in:,}  outside {n_out:,}  |  years {YR0}–{YR1}')
print(pd.DataFrame({'n': n_r,
                    'peak': [MONTH_LAB[int(i) - 1] for i in S.idxmax(axis=1)],
                    'peak %': S.max(axis=1).round(1),
                    'min %':  S.min(axis=1).round(1)}).to_string())
print('\nSub-region counts\n' + D['subregion'].value_counts().to_string())

S.round(3).to_csv(os.path.join(FIGDIR, 'fig1d_month_share.csv'))          # raw data for SI table
O.astype(int).to_csv(os.path.join(FIGDIR, 'fig1d_month_counts.csv'))

# ══════════════ 3. Plotting ══════════════
CMAP_D = LinearSegmentedColormap.from_list(       # low-saturation blue-white-ochre diverging colormap
    'dev_bwr', ['#2f6fae', '#7ba7cd', '#d7e3ec', '#f6f7f8',
                '#f0d5cd', '#dd9d8f', '#c4614f'])

def draw_d(annot):
    V    = S.values                      # share (%) -> cell numbers
    DEV  = V - UNIF                      # deviation (pp) -> color
    vmax = max(2.0, float(np.ceil(np.nanpercentile(np.abs(DEV), 98))))
    norm = Normalize(-vmax, vmax)
    cmap = CMAP_D.copy(); cmap.set_bad('#f2f3f4')

    fig = plt.figure(figsize=(10.6, 6.4), dpi=500)
    ax  = fig.add_axes([0.235, 0.245, 0.700, 0.640])
    ax.pcolormesh(np.arange(13), np.arange(len(keys) + 1), np.ma.masked_invalid(DEV),
                  cmap=cmap, norm=norm, edgecolors='white', linewidth=3.0, zorder=1)
    ax.invert_yaxis()
    ax.set_xticks(np.arange(12) + .5)
    ax.set_xticklabels(MONTH_LAB, fontsize=9, fontweight='bold', color=INK)
    ax.set_yticks(np.arange(len(keys)) + .5)
    ax.set_yticklabels([n for _, n in ROW_META], fontsize=9.5, fontweight='bold', color=INK)
    ax.tick_params(length=0, pad=6)
    for sp in ax.spines.values():
        sp.set_visible(False)

    if SHOW_VAL:
        for i in range(V.shape[0]):
            for j in range(12):
                if np.isfinite(V[i, j]):
                    ax.text(j + .5, i + .5, f'{V[i, j]:.1f}', ha='center', va='center',
                            fontsize=6.8, zorder=3,
                            color='white' if abs(DEV[i, j]) > .78 * vmax else INK)

    if annot:
        for i, r in enumerate(keys):
            ax.text(12.15, i + .5, f'n = {n_r[r]:,}', ha='left', va='center',
                    fontsize=8.2, color=INK2)

    cax = fig.add_axes([0.265, 0.095, 0.320, 0.020])
    cb  = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax,
                       orientation='horizontal', ticks=[-vmax, 0, vmax])
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0, pad=3, labelsize=7.6, colors=INK2)
    cb.set_ticklabels([f'−{vmax:.0f}', '0', f'+{vmax:.0f}'])

    if annot:
        fig.text(0.030, 0.955, 'd', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.090, 0.958, 'Monthly distribution of accidents by region',
                 fontsize=12.5, fontweight='bold', color=INK)
        fig.text(0.605, 0.103, 'Deviation from an even month\n(percentage points)',
                 fontsize=8.6, fontweight='bold', color=INK, va='center', linespacing=1.4)
        fig.text(0.030, 0.050,
                 f'Cell numbers give each month’s share of the region’s {YR0}–{YR1} cargo-ship '
                 'accidents (rows sum to 100%); colour shows\nthe departure from an even '
                 f'distribution ({UNIF:.1f}% per month). Regions are fixed latitude–longitude '
                 'masks; counts are not\nadjusted for seasonal variation in traffic volume.',
                 fontsize=8.0, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_d, 'fig1d_month_share')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel d: region x month accident-share heatmap
#  Color = deviation from uniform month (pp, diverging colormap, white = 8.3%)
#  Number = month's share of the region's accidents (%), rows sum to 100%
#  Rows grouped by storm regime: extratropical winter -> two-season -> tropical
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

INK, INK2, GREY = '#1a1a1a', '#5a5f66', '#9aa0a6'
MONTH_LAB = list('JFMAMJJASOND')
NORM_BY   = 'row'      # 'row' = share of region total (rows sum 100%); 'all' = share of grand total (table sums 100%)
SHOW_VAL  = True       # annotate values in cells
SHOW_BOX  = True       # draw dashed season-window boxes

# ══════════════ 1. Region boundaries (verbatim from M2_merge CELL 1)══════════════
# Structure: region -> subregion -> [lat/lon rectangles...], written (S, N, W, E)
# Rule: lat ∈ [S, N) and lon ∈ [W, E), always half-open; multiple rectangles of a subregion are unioned
REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic': [(36, 45, -15, -6)],
        'Biscay_Celtic':   [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':       [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}

# row order = storm-regime grouping, not geographic
ROW_META = [('NW_Europe',      'NW Europe'),               # ── extratropical winter storms dominate
            ('Baltic',         'Baltic Sea'),
            ('Med_Black',      'Mediterranean–Black Sea'),
            ('NW_Atlantic',    'NW Atlantic'),             # ── two seasons: winter storms + hurricanes
            ('Asia',           'E & SE Asia'),             # ── tropical cyclones dominate
            ('Indian_Ocean_N', 'N Indian Ocean')]          # ── monsoon + bimodal cyclones
keys = [k for k, _ in ROW_META]
assert set(keys) == set(REGIONS), 'row order inconsistent with region definitions'

# Season windows: external climatological definitions fixed before data; visual reference only, not used in stats
# Winter is NDJFM (not ONDJFM) to avoid overlapping the NW Atlantic hurricane box in October
SEASON_BOXES = [
    dict(regions=['NW_Europe', 'Baltic', 'Med_Black', 'NW_Atlantic'],
         spans=[(1, 3), (11, 12)], label='extratropical winter storms (NDJFM)'),
    dict(regions=['NW_Atlantic'],    spans=[(8, 10)], label='Atlantic hurricanes (Aug–Oct)'),
    dict(regions=['Asia'],           spans=[(6, 11)], label='NW Pacific typhoons (Jun–Nov)'),
    dict(regions=['Indian_Ocean_N'], spans=[(6, 9)],  label='SW monsoon (Jun–Sep)'),
]

def region_of(lat, lon):
    """Vectorized region assignment. Returns (region, subregion) object arrays; None outside.
       A point in multiple boxes goes to the first hit in REGIONS order, with a warning."""
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    reg    = np.full(lat.shape, None, object)
    sub    = np.full(lat.shape, None, object)
    filled = np.zeros(lat.shape, bool)
    nhit   = np.zeros(lat.shape, int)
    for r, subs in REGIONS.items():
        for s, boxes in subs.items():
            hit = np.zeros(lat.shape, bool)
            for (S, N, W, E) in boxes:
                hit |= (lat >= S) & (lat < N) & (lon >= W) & (lon < E)
            nhit += hit
            take = hit & ~filled
            reg[take], sub[take] = r, s
            filled |= take
    n_ov = int((nhit > 1).sum())
    if n_ov:
        print(f'⚠ {n_ov} points fall in multiple boxes (assigned by definition order); check for overlapping boundaries')
    return reg, sub

def _runs(idx):
    """Split row indices into contiguous runs; dashed boxes only span adjacent rows"""
    out, s = [], sorted(idx)
    while s:
        a = b = s.pop(0)
        while s and s[0] == b + 1:
            b = s.pop(0)
        out.append((a, b + 1))
    return out

print('Region boundaries (lat ∈ [S,N), lon ∈ [W,E))')
for r, _ in ROW_META:
    print(f'  {r}')
    for s, boxes in REGIONS[r].items():
        print(f'    {s:<16s} ' + '  ∪  '.join(f'{S}–{N}N, {W}–{E}E' for S, N, W, E in boxes))

# ══════════════ 2. Accident region assignment and monthly stats ══════════════
_mcol = next((c for c in ['acc_month', 'month', 'casualty_month'] if c in acc.columns), None)
if _mcol:
    _mo = pd.to_numeric(acc[_mcol], errors='coerce')
else:
    _dcol = next(c for c in ['acc_date', 'casualty_date', 'date'] if c in acc.columns)
    _mo = pd.to_datetime(acc[_dcol], errors='coerce').dt.month

D = pd.DataFrame({
    'lat':   pd.to_numeric(acc['lat_dec'],  errors='coerce'),
    'lon':   pd.to_numeric(acc['lon_dec'],  errors='coerce'),
    'month': _mo,
    'year':  pd.to_numeric(acc['acc_year'], errors='coerce'),
}).dropna(subset=['lat', 'lon', 'month', 'year'])
D = D[D['year'].between(YR_MIN, YR_MAX)].copy()
D['region'], D['subregion'] = region_of(D['lat'].values, D['lon'].values)

n_in, n_out = int(D['region'].notna().sum()), int(D['region'].isna().sum())
D = D.dropna(subset=['region'])
YR0, YR1 = int(D['year'].min()), int(D['year'].max())

O    = (pd.crosstab(D['region'], D['month'].astype(int))
          .reindex(keys).reindex(columns=range(1, 13)).fillna(0.0))
S    = O.div(O.sum(axis=1), axis=0) * 100 if NORM_BY == 'row' else O / O.values.sum() * 100
n_r  = O.sum(axis=1).astype(int)
UNIF = 100 / 12

print(f'\naccidents: in regions {n_in:,}  outside {n_out:,}  |  years {YR0}–{YR1}')
print(pd.DataFrame({'n': n_r,
                    'peak':   [MONTH_LAB[int(i) - 1] for i in S.idxmax(axis=1)],
                    'peak %': S.max(axis=1).round(1),
                    'min %':  S.min(axis=1).round(1)}).to_string())
print('\nSub-region counts\n' + D['subregion'].value_counts().to_string())

S.round(3).to_csv(os.path.join(FIGDIR, 'fig1d_month_share.csv'))       # raw data for SI table
O.astype(int).to_csv(os.path.join(FIGDIR, 'fig1d_month_counts.csv'))

# ══════════════ 3. Plotting ══════════════
CMAP_D = LinearSegmentedColormap.from_list(       # low-saturation blue-white-ochre diverging colormap
    'dev_bwr', ['#2f6fae', '#7ba7cd', '#d7e3ec', '#f6f7f8',
                '#f0d5cd', '#dd9d8f', '#c4614f'])

def draw_d(annot):
    V    = S.values                      # share (%) -> cell numbers
    DEV  = V - UNIF                      # deviation (pp) -> color
    vmax = max(2.0, float(np.ceil(np.nanpercentile(np.abs(DEV), 98))))
    norm = Normalize(-vmax, vmax)
    cmap = CMAP_D.copy(); cmap.set_bad('#f2f3f4')

    fig = plt.figure(figsize=(10.6, 6.6), dpi=500)
    ax  = fig.add_axes([0.235, 0.250, 0.700, 0.635])
    ax.pcolormesh(np.arange(13), np.arange(len(keys) + 1), np.ma.masked_invalid(DEV),
                  cmap=cmap, norm=norm, edgecolors='white', linewidth=3.0, zorder=1)
    ax.invert_yaxis()
    ax.set_xticks(np.arange(12) + .5)
    ax.set_xticklabels(MONTH_LAB, fontsize=9, fontweight='bold', color=INK)
    ax.set_yticks(np.arange(len(keys)) + .5)
    ax.set_yticklabels([n for _, n in ROW_META], fontsize=9.5, fontweight='bold', color=INK)
    ax.tick_params(length=0, pad=6)
    for sp in ax.spines.values():
        sp.set_visible(False)

    if SHOW_VAL:
        for i in range(V.shape[0]):
            for j in range(12):
                if np.isfinite(V[i, j]):
                    ax.text(j + .5, i + .5, f'{V[i, j]:.1f}', ha='center', va='center',
                            fontsize=6.8, zorder=3,
                            color='white' if abs(DEV[i, j]) > .78 * vmax else INK)

    if SHOW_BOX:
        for box in SEASON_BOXES:
            rows  = _runs([keys.index(r) for r in box['regions'] if r in keys])
            first = True
            for (m0, m1) in box['spans']:
                for (r0, r1) in rows:
                    ax.add_patch(mpatches.Rectangle((m0 - 1, r0), m1 - m0 + 1, r1 - r0,
                                 fill=False, ec=GREY, lw=.9, ls=(0, (4, 3)), zorder=4))
                    if annot and first:
                        ax.text((m0 - 1 + m1) / 2, r1 - .22, box['label'], ha='center',
                                va='center', fontsize=7.2, color=INK2, zorder=5)
                        first = False

    if annot:
        for i, r in enumerate(keys):
            ax.text(12.15, i + .5, f'n = {n_r[r]:,}', ha='left', va='center',
                    fontsize=8.2, color=INK2)

    cax = fig.add_axes([0.265, 0.100, 0.320, 0.020])
    cb  = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax,
                       orientation='horizontal', ticks=[-vmax, 0, vmax])
    cb.outline.set_visible(True)                                   # show colorbar outline
    cb.minorticks_off()                                  # minor ticks off
    cb.ax.xaxis.set_ticks_position('none')               # tick marks off
    cb.ax.tick_params(which='both', length=0, width=0, pad=4,
                      labelsize=7.6, colors=INK2)
    cb.set_ticklabels([f'−{vmax:.0f}', '0', f'+{vmax:.0f}'])


    if annot:
        fig.text(0.030, 0.955, 'd', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.090, 0.958, 'Monthly distribution of accidents by region',
                 fontsize=12.5, fontweight='bold', color=INK)
        fig.text(0.605, 0.108, 'Deviation from an even month\n(percentage points)',
                 fontsize=8.6, fontweight='bold', color=INK, va='center', linespacing=1.4)
        fig.text(0.030, 0.052,
                 f'Cell numbers give each month’s share of the region’s {YR0}–{YR1} cargo-ship '
                 'accidents (rows sum to 100%); colour shows the departure\nfrom an even '
                 f'distribution ({UNIF:.1f}% per month). Rows are ordered by storm regime. Dashed '
                 'windows mark seasons defined a priori from basin\nclimatology (North Atlantic and '
                 'Mediterranean cyclone tracks; Atlantic hurricane and NW Pacific typhoon seasons; '
                 'Indian Ocean SW monsoon)\nand are not fitted to these data. Counts are not '
                 'adjusted for seasonal variation in traffic volume.',
                 fontsize=8.0, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_d, 'fig1d_month_share')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Panel d: region x month accident-share heatmap
#  Color = deviation from uniform month (pp, diverging colormap, white = 8.3%)
#  Number = month's share of the region's accidents (%), rows sum to 100%
#  Rows grouped by storm regime: extratropical winter -> two-season -> tropical
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

INK, INK2, GREY = '#1a1a1a', '#5a5f66', '#9aa0a6'
MONTH_LAB = list('JFMAMJJASOND')
NORM_BY   = 'row'      # 'row' = share of region total (rows sum 100%); 'all' = share of grand total (table sums 100%)
SHOW_VAL  = True       # annotate values in cells
SHOW_BOX  = True       # draw dashed season-window boxes

# ══════════════ 1. Region boundaries (verbatim from M2_merge CELL 1)══════════════
# Structure: region -> subregion -> [lat/lon rectangles...], written (S, N, W, E)
# Rule: lat ∈ [S, N) and lon ∈ [W, E), always half-open; multiple rectangles of a subregion are unioned
REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic': [(36, 45, -15, -6)],
        'Biscay_Celtic':   [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':       [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}

# row order = storm-regime grouping, not geographic
ROW_META = [('NW_Europe',      'NW Europe'),               # ── extratropical winter storms dominate
            ('Baltic',         'Baltic Sea'),
            ('Med_Black',      'Mediterranean–Black Sea'),
            ('NW_Atlantic',    'NW Atlantic'),             # ── two seasons: winter storms + hurricanes
            ('Asia',           'E & SE Asia'),             # ── tropical cyclones dominate
            ('Indian_Ocean_N', 'N Indian Ocean')]          # ── monsoon + bimodal cyclones
keys = [k for k, _ in ROW_META]
assert set(keys) == set(REGIONS), 'row order inconsistent with region definitions'

# Season windows: external climatological definitions fixed before data; visual reference only, not used in stats
# Winter is NDJFM (not ONDJFM) to avoid overlapping the NW Atlantic hurricane box in October
SEASON_BOXES = [
    dict(regions=['NW_Europe', 'Baltic', 'Med_Black', 'NW_Atlantic'],
         spans=[(1, 3), (11, 12)], label='extratropical winter storms (NDJFM)'),
    dict(regions=['NW_Atlantic'],    spans=[(8, 10)], label='Atlantic hurricanes (Aug–Oct)'),
    dict(regions=['Asia'],           spans=[(6, 11)], label='NW Pacific typhoons (Jun–Nov)'),
    dict(regions=['Indian_Ocean_N'], spans=[(6, 9)],  label='SW monsoon (Jun–Sep)'),
]

def region_of(lat, lon):
    """Vectorized region assignment. Returns (region, subregion) object arrays; None outside.
       A point in multiple boxes goes to the first hit in REGIONS order, with a warning."""
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    reg    = np.full(lat.shape, None, object)
    sub    = np.full(lat.shape, None, object)
    filled = np.zeros(lat.shape, bool)
    nhit   = np.zeros(lat.shape, int)
    for r, subs in REGIONS.items():
        for s, boxes in subs.items():
            hit = np.zeros(lat.shape, bool)
            for (S, N, W, E) in boxes:
                hit |= (lat >= S) & (lat < N) & (lon >= W) & (lon < E)
            nhit += hit
            take = hit & ~filled
            reg[take], sub[take] = r, s
            filled |= take
    n_ov = int((nhit > 1).sum())
    if n_ov:
        print(f'⚠ {n_ov} points fall in multiple boxes (assigned by definition order); check for overlapping boundaries')
    return reg, sub

def _runs(idx):
    """Split row indices into contiguous runs; dashed boxes only span adjacent rows"""
    out, s = [], sorted(idx)
    while s:
        a = b = s.pop(0)
        while s and s[0] == b + 1:
            b = s.pop(0)
        out.append((a, b + 1))
    return out

print('Region boundaries (lat ∈ [S,N), lon ∈ [W,E))')
for r, _ in ROW_META:
    print(f'  {r}')
    for s, boxes in REGIONS[r].items():
        print(f'    {s:<16s} ' + '  ∪  '.join(f'{S}–{N}N, {W}–{E}E' for S, N, W, E in boxes))

# ══════════════ 2. Accident region assignment and monthly stats ══════════════
_mcol = next((c for c in ['acc_month', 'month', 'casualty_month'] if c in acc.columns), None)
if _mcol:
    _mo = pd.to_numeric(acc[_mcol], errors='coerce')
else:
    _dcol = next(c for c in ['acc_date', 'casualty_date', 'date'] if c in acc.columns)
    _mo = pd.to_datetime(acc[_dcol], errors='coerce').dt.month

D = pd.DataFrame({
    'lat':   pd.to_numeric(acc['lat_dec'],  errors='coerce'),
    'lon':   pd.to_numeric(acc['lon_dec'],  errors='coerce'),
    'month': _mo,
    'year':  pd.to_numeric(acc['acc_year'], errors='coerce'),
}).dropna(subset=['lat', 'lon', 'month', 'year'])
D = D[D['year'].between(YR_MIN, YR_MAX)].copy()
D['region'], D['subregion'] = region_of(D['lat'].values, D['lon'].values)

n_in, n_out = int(D['region'].notna().sum()), int(D['region'].isna().sum())
D = D.dropna(subset=['region'])
YR0, YR1 = int(D['year'].min()), int(D['year'].max())

O    = (pd.crosstab(D['region'], D['month'].astype(int))
          .reindex(keys).reindex(columns=range(1, 13)).fillna(0.0))
S    = O.div(O.sum(axis=1), axis=0) * 100 if NORM_BY == 'row' else O / O.values.sum() * 100
n_r  = O.sum(axis=1).astype(int)
UNIF = 100 / 12

print(f'\naccidents: in regions {n_in:,}  outside {n_out:,}  |  years {YR0}–{YR1}')
print(pd.DataFrame({'n': n_r,
                    'peak':   [MONTH_LAB[int(i) - 1] for i in S.idxmax(axis=1)],
                    'peak %': S.max(axis=1).round(1),
                    'min %':  S.min(axis=1).round(1)}).to_string())
print('\nSub-region counts\n' + D['subregion'].value_counts().to_string())

S.round(3).to_csv(os.path.join(FIGDIR, 'fig1d_month_share.csv'))       # raw data for SI table
O.astype(int).to_csv(os.path.join(FIGDIR, 'fig1d_month_counts.csv'))

# ══════════════ 3. Plotting ══════════════
CMAP_D = LinearSegmentedColormap.from_list(       # low-saturation blue-white-ochre diverging colormap
    'dev_bwr', ['#2f6fae', '#7ba7cd', '#d7e3ec', '#f6f7f8',
                '#f0d5cd', '#dd9d8f', '#c4614f'])

def draw_d(annot):
    V    = S.values                      # share (%) -> cell numbers
    DEV  = V - UNIF                      # deviation (pp) -> color
    vmax = max(2.0, float(np.ceil(np.nanpercentile(np.abs(DEV), 98))))
    norm = Normalize(-vmax, vmax)
    cmap = CMAP_D.copy(); cmap.set_bad('#f2f3f4')

    fig = plt.figure(figsize=(10.6, 6.6), dpi=500)
    ax  = fig.add_axes([0.235, 0.250, 0.700, 0.635])
    ax.pcolormesh(np.arange(13), np.arange(len(keys) + 1), np.ma.masked_invalid(DEV),
                  cmap=cmap, norm=norm, edgecolors='white', linewidth=3.0, zorder=1)
    ax.invert_yaxis()
    ax.set_xticks(np.arange(12) + .5)
    ax.set_xticklabels(MONTH_LAB, fontsize=9, fontweight='bold', color=INK)
    ax.set_yticks(np.arange(len(keys)) + .5)
    ax.set_yticklabels([n for _, n in ROW_META], fontsize=9.5, fontweight='bold', color=INK)
    ax.tick_params(length=0, pad=6)
    for sp in ax.spines.values():
        sp.set_visible(False)

    # if SHOW_VAL:
    #     for i in range(V.shape[0]):
    #         for j in range(12):
    #             if np.isfinite(V[i, j]):
    #                 ax.text(j + .5, i + .5, f'{V[i, j]:.1f}', ha='center', va='center',
    #                         fontsize=6.8, zorder=3,
    #                         color='white' if abs(DEV[i, j]) > .78 * vmax else INK)

    # if SHOW_BOX:
    #     for box in SEASON_BOXES:
    #         rows  = _runs([keys.index(r) for r in box['regions'] if r in keys])
    #         first = True
    #         for (m0, m1) in box['spans']:
    #             for (r0, r1) in rows:
    #                 ax.add_patch(mpatches.Rectangle((m0 - 1, r0), m1 - m0 + 1, r1 - r0,
    #                              fill=False, ec=GREY, lw=.9, ls=(0, (4, 3)), zorder=4))
    #                 if annot and first:
    #                     ax.text((m0 - 1 + m1) / 2, r1 - .22, box['label'], ha='center',
    #                             va='center', fontsize=7.2, color=INK2, zorder=5)
    #                     first = False

    # if annot:
    #     for i, r in enumerate(keys):
    #         ax.text(12.15, i + .5, f'n = {n_r[r]:,}', ha='left', va='center',
    #                 fontsize=8.2, color=INK2)

    cax = fig.add_axes([0.265, 0.100, 0.320, 0.020])
    cb  = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax,
                       orientation='horizontal', ticks=[-vmax, 0, vmax])
    cb.outline.set_visible(True)                                   # show colorbar outline
    cb.minorticks_off()                                  # minor ticks off
    cb.ax.xaxis.set_ticks_position('none')               # tick marks off
    cb.ax.tick_params(which='both', length=0, width=0, pad=4,
                      labelsize=7.6, colors=INK2)
    cb.set_ticklabels([f'−{vmax:.0f}', '0', f'+{vmax:.0f}'])


    if annot:
        fig.text(0.030, 0.955, 'd', fontsize=17, fontweight='bold', color=INK)
        fig.text(0.090, 0.958, 'Monthly distribution of accidents by region',
                 fontsize=12.5, fontweight='bold', color=INK)
        fig.text(0.605, 0.108, 'Deviation from an even month\n(percentage points)',
                 fontsize=8.6, fontweight='bold', color=INK, va='center', linespacing=1.4)
        fig.text(0.030, 0.052,
                 f'Cell numbers give each month’s share of the region’s {YR0}–{YR1} cargo-ship '
                 'accidents (rows sum to 100%); colour shows the departure\nfrom an even '
                 f'distribution ({UNIF:.1f}% per month). Rows are ordered by storm regime. Dashed '
                 'windows mark seasons defined a priori from basin\nclimatology (North Atlantic and '
                 'Mediterranean cyclone tracks; Atlantic hurricane and NW Pacific typhoon seasons; '
                 'Indian Ocean SW monsoon)\nand are not fitted to these data. Counts are not '
                 'adjusted for seasonal variation in traffic volume.',
                 fontsize=8.0, color=INK2, va='top', linespacing=1.5)
    return fig

save_two(draw_d, 'fig1d_month_share')


In [ ]:
# ================= Block 8: 2015-2020 same-period validation (with significance) =================
def build_A(y0, y1):
    laV = pd.to_numeric(acc["lat_dec"], errors="coerce")
    loV = pd.to_numeric(acc["lon_dec"], errors="coerce")
    yrV = pd.to_numeric(acc["acc_year"], errors="coerce")
    okV = laV.notna() & loV.notna() & yrV.between(y0, y1)
    lonb = np.clip(np.floor(loV[okV].values + 180).astype(int), 0, NLON - 1)
    latb = np.clip(np.floor(laV[okV].values + 90).astype(int),  0, NLAT - 1)
    return np.bincount(latb * NLON + lonb, minlength=NLAT * NLON).reshape(NLAT, NLON)

def star(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"

def fit_period(y0, y1):
    Ag = build_A(y0, y1)
    rr2, cc2 = np.where(domain)
    dd = pd.DataFrame({"E": E_grid[rr2, cc2] / YEARS, "A": Ag[rr2, cc2].astype(int)})
    dd["logE"] = np.log(dd["E"])
    mfit = smf.glm("A ~ logE", data=dd, family=sm.families.Poisson()).fit(cov_type="HC1")
    b, s = mfit.params["logE"], mfit.bse["logE"]
    z1 = (b - 1) / s; p1 = 2 * (1 - norm.cdf(abs(z1)))
    rho2, psp2 = spearmanr(dd["E"], dd["A"])
    return {"period": f"{y0}-{y1}", "cells": len(dd), "accidents": int(dd["A"].sum()),
            "beta": round(b, 3), "SE": round(s, 3),
            "beta_95CI": f"[{b-1.96*s:.3f},{b+1.96*s:.3f}]",
            "p(β=1)": f"{p1:.2g}{star(p1)}", "rho": round(rho2, 3)}

comp = pd.DataFrame([fit_period(2002, 2021), fit_period(2015, 2020)])
print(comp.to_string(index=False))


In [ ]:
# ================= Fig S1 (SI): accident composition x risk bins (near-shore/offshore strata, two versions) =================
# Purpose: show cross-sectional excess risk is confounded by coverage bias/congestion/piracy and cannot isolate weather -> motivates the M2 design
land_any = np.zeros((NLAT, NLON), bool)
for dla in (-0.35, 0.0, 0.35):
    for dlo in (-0.35, 0.0, 0.35):
        land_any |= globe.is_land(LatC + dla, LonC + dlo)
coastal_grid = ndimage.binary_dilation(land_any, iterations=1)
df["coastal"] = coastal_grid[df["lat_bin"].values, df["lon_bin"].values]

laS = pd.to_numeric(acc["lat_dec"], errors="coerce")
loS = pd.to_numeric(acc["lon_dec"], errors="coerce")
yrS = pd.to_numeric(acc["acc_year"], errors="coerce")
okS = laS.notna() & loS.notna() & yrS.between(YR_MIN, YR_MAX)
ev = acc.loc[okS].copy()
ev["lat_bin"] = np.clip(np.floor(laS[okS] + 90).astype(int), 0, NLAT - 1)
ev["lon_bin"] = np.clip(np.floor(loS[okS] + 180).astype(int), 0, NLON - 1)
ev = ev.merge(df[["lat_bin", "lon_bin", "oe_eb", "coastal"]], on=["lat_bin", "lon_bin"], how="inner")

cand = [c for c in ["cause", "Cause of Casualty", "cause_of_casualty",
                    "casualty_cause", "acc_cause"] if c in ev.columns]
assert cand, f"accident cause column not found: {list(ev.columns)}"
cz = ev[cand[0]].fillna("").astype(str).str.lower()
TYPE_FILTERS = {                                   # same regex set as M2/M4
    'Collision':      r'collision|contact',
    'Stranding':      r'strand|ground|shoal|beach',
    'Machinery':      r'machin|engine|propul',
    'Foundering':     r'founder|submer',
    'Fire_Explosion': r'fire|explo|burn',
    'Hull_Damage':    r'hull|holed|crack|structur',
    'Piracy':         r'piracy',
}
flags = pd.DataFrame({k: cz.str.contains(v) for k, v in TYPE_FILTERS.items()})
PRIORITY = ['Foundering', 'Stranding', 'Fire_Explosion', 'Collision',
            'Hull_Damage', 'Machinery', 'Piracy']
ev["cat"] = 'Other'
for k in reversed(PRIORITY):
    ev.loc[flags[k].values, "cat"] = k

SHOW = {"Foundering":     ("#b2182b", "o"),
        "Stranding":      ("#ef8a62", "s"),
        "Machinery":      ("#c51b7d", "^"),
        "Collision":      ("#2166ac", "v"),
        "Fire_Explosion": ("#666666", "x"),
        "Piracy":         ("#999999", "+")}

def draw_s1(annot):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    for ax, (name, sub) in zip(axes, [("offshore", ev[~ev.coastal]), ("coastal", ev[ev.coastal])]):
        sub = sub.copy()
        sub["q"] = pd.qcut(sub["oe_eb"].rank(method="first"), 5, labels=False) + 1
        med = sub.groupby("q")["oe_eb"].median()
        compt = sub.groupby(["q", "cat"]).size().unstack(fill_value=0)
        share = compt.div(compt.sum(axis=1), axis=0); nq = compt.sum(axis=1)
        for cat, (col, mk) in SHOW.items():
            if cat in share.columns:
                y = share[cat] * 100
                ci = np.sqrt(share[cat] * (1 - share[cat]) / nq) * 196
                ax.errorbar(share.index, y, yerr=ci, color=col, marker=mk,
                            ms=5, lw=1.6, capsize=2, label=cat)
        ax.set_xticks(range(1, 6))
        if annot:
            ax.set_xticklabels([f"Q{i}\nO/E≈{med.get(i, np.nan):.2f}" for i in range(1, 6)], fontsize=8)
            ax.set_xlabel("risk bins (equal accident counts per bin)")
            ax.set_title(f"{name}   (n={len(sub):,})")
        else:
            ax.set_xticklabels([str(i) for i in range(1, 6)])
        ax.grid(alpha=0.25, lw=0.4)
    if annot:
        axes[0].set_ylabel("share of accidents in bin (%)")
        axes[0].legend(frameon=False, fontsize=8)
        fig.suptitle("S1  accident-type composition across risk bins, by coastal proximity", y=1.02)
    return fig

save_two(draw_s1, "figS1_composition")
